In [1]:
import pandas as pd
import numpy as np
import io
df = pd.read_csv('amazon_cleaned.csv', sep=',')

In [2]:
print(f"""
Correlation Coefficients Matrix 
Allows us to identify correlations between our input features of interest to check for multicollinearity, which is helpful because we can understand which features are statistically significant and truly drive the results (i.e., what impacts how a product is considered a “deal”).
Outlier Detection for Price (i.e., IQR or z-score)
Model types like KNN and Multiple Logistic Regression can be sensitive to extreme values due to distance distortion or coefficient leverage, respectively. Identifying these ahead of time gives us the ability to account for them before they potentially cause issues for these models later.
Category Cardinality and Per-Category Deal Rate
We need to know whether “deal” is a category-specific phenomenon: some categories (e.g., high-margin categories such as electronics accessories) may routinely offer high discounts where others do not. A tool which is accurate overall but poor within a specific category that a shopper cares about isn’t actually useful to them. For modeling, this can especially affect KNN (depending on encoding) and Logistic Regression (due to potentially unstable coefficients for rare categories) model types, but Random Forest and Decision Tree model types can handle sparse categorical splits more gracefully and can naturally rank category importance.
""")


Correlation Coefficients Matrix 
Allows us to identify correlations between our input features of interest to check for multicollinearity, which is helpful because we can understand which features are statistically significant and truly drive the results (i.e., what impacts how a product is considered a “deal”).
Outlier Detection for Price (i.e., IQR or z-score)
Model types like KNN and Multiple Logistic Regression can be sensitive to extreme values due to distance distortion or coefficient leverage, respectively. Identifying these ahead of time gives us the ability to account for them before they potentially cause issues for these models later.
Category Cardinality and Per-Category Deal Rate
We need to know whether “deal” is a category-specific phenomenon: some categories (e.g., high-margin categories such as electronics accessories) may routinely offer high discounts where others do not. A tool which is accurate overall but poor within a specific category that a shopper cares about 

In [3]:
df.corr(numeric_only=True)

,discounted_price,actual_price,discount_percentage,rating,rating_count,is_deal
discounted_price,1.000000,0.961906,-0.241969,0.114947,-0.027261,0.015606
actual_price,0.961906,1.000000,-0.117494,0.117315,-0.036137,0.083990
discount_percentage,-0.241969,-0.117494,1.000000,-0.132601,0.011691,0.343482
rating,0.114947,0.117315,-0.132601,1.000000,0.099549,0.505593
rating_count,-0.027261,-0.036137,0.011691,0.099549,1.000000,0.114520
is_deal,0.015606,0.083990,0.343482,0.505593,0.114520,1.000000


In [4]:
print(df.quantile(0.75, numeric_only=True) - df.quantile(0.25, numeric_only=True))

discounted_price        1674.0
actual_price            3512.5
discount_percentage       31.0
rating                     0.3
rating_count           16150.5
is_deal                    1.0
dtype: float64


In [22]:
#df['category'] = df['category'].str.split('|', expand=True)[0]
categories_to_drop = ['MusicalInstruments', 'HomeImprovement', 'Toys&Games', 'Car&Motorbike', 'Health&PersonalCare']
df = df[~df['category'].isin(categories_to_drop)]

deals_df = df[(df['is_deal'] == True)]

print("Overall Count:")
print(df['category'].value_counts())
print("\nDeal Count:")
print(deals_df['category'].value_counts())

Overall Count:
category
Electronics              526
Computers&Accessories    451
Home&Kitchen             448
OfficeProducts            31
Name: count, dtype: int64

Deal Count:
category
Computers&Accessories    272
Electronics              246
Home&Kitchen             165
OfficeProducts             4
Name: count, dtype: int64


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn.metrics as metrics

x = df[['category','discounted_price','actual_price','rating_count']].copy()
y = df['is_deal']

x = pd.get_dummies(x, columns=['category'], drop_first=True)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2, stratify=y)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

model = LogisticRegression()
model.fit(x_train_scaled, y_train)

y_prediction = model.predict(x_test_scaled)
y_probabilities = model.predict_proba(x_test_scaled)[:, 1]

print(f"Accuracy: {metrics.accuracy_score(y_test, y_prediction)}")
print(f"F1-Score: {metrics.f1_score(y_test, y_prediction)}")
print(f"ROC-AUC Score: {metrics.roc_auc_score(y_test, y_probabilities)}")
print(f"Classification Report:\n{metrics.classification_report(y_test, y_prediction)}")

Accuracy: 0.5993150684931506
F1-Score: 0.5517241379310345
ROC-AUC Score: 0.6795125164690381
Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.67      0.64       154
           1       0.59      0.52      0.55       138

    accuracy                           0.60       292
   macro avg       0.60      0.60      0.59       292
weighted avg       0.60      0.60      0.60       292

